# SDK-Sovereign Hardened Training Pipeline

All-in-one notebook with **aggressive logging and auto-backup** so that disconnects don't lose work.

## What this notebook does differently from the previous one

1. **Localhost env server** — no WebSocket timeouts to remote HF Space during training
2. **HF Hub auto-backup every N steps** — even if Colab disconnects, latest checkpoint is on Hub
3. **Verbose JSONL logs** — every action, every reward, every step saved locally to disk
4. **W&B with system metrics** — GPU memory, CPU, disk usage tracked
5. **Resume support** — re-running this notebook continues from latest HF Hub checkpoint
6. **Eval baseline before training** — establishes baseline metrics in W&B

## Required Colab Secrets (Tools menu → Secrets)
- `HF_TOKEN` (write-scoped, must have access to gated Llama models if using them)
- `WANDB_API_KEY`

## Runtime
- GPU: T4 (free tier OK, A10G better if available)
- Run cells top to bottom
- Total time: ~2.5-3 hours on T4

## What gets saved permanently (survives disconnect)
- W&B run at `https://wandb.ai/<your-entity>/sdk-sovereign-hardened`
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-checkpoints` — full checkpoints every save_steps
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-lead-adapter` — final Lead adapter
- HF Hub repo at `https://huggingface.co/<HF_USER>/sdk-sovereign-auditor-adapter` — final Auditor adapter


## Cell 1 — Install dependencies

In [ ]:
# Cell 1 — Pinned dependency stack
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q -U "trl==0.24.0" "datasets==3.6.0" "accelerate==1.6.0" "pydantic==2.10.6" peft bitsandbytes
!pip install -q "openenv-core[core] @ git+https://github.com/meta-pytorch/OpenEnv@v0.2.3"
!pip install -q wandb huggingface_hub matplotlib numpy psutil

print('OK: dependencies installed')

## Cell 2 — Configuration and authentication

In [ ]:
# Cell 2 — Configuration, secrets, repo clone
import os
import sys
import json
import warnings
import subprocess
from pathlib import Path
from datetime import datetime

from google.colab import userdata
from huggingface_hub import HfApi, create_repo, login

warnings.filterwarnings("ignore", message="Both `max_new_tokens`")
warnings.filterwarnings("ignore", category=FutureWarning)

# === KEEP THIS STRUCTURE THE SAME; EDIT VALUES ONLY ===
GH_REPO = "ishansurdi/SDK-Sovereign"
HF_USER = "ishansurdi"
WANDB_PROJECT = "sdk-sovereign-fast"
WANDB_ENTITY = ""

# === HF Hub repos ===
CHECKPOINT_REPO = f"{HF_USER}/sdk-sovereign-checkpoints"
LEAD_ADAPTER_REPO = f"{HF_USER}/sdk-sovereign-lead-adapter"
AUDITOR_ADAPTER_REPO = f"{HF_USER}/sdk-sovereign-auditor-adapter"

# === Fast-run scale ===
EXPERT_EPISODES_PER_REPO = 8
NEGATIVE_AUDITOR_EXAMPLES_PER_REPO = 4
N_BASELINE_EVAL_PER_REPO = 4
N_TRAINED_EVAL_PER_REPO = 6
SAVE_EVERY_N_STEPS = 10
LOG_EVERY_N_STEPS = 1
LEAD_SFT_MAX_STEPS = 40
AUDITOR_SFT_MAX_STEPS = 28
LEAD_GRPO_MAX_STEPS = 12
AUDITOR_GRPO_MAX_STEPS = 8

# === Phase toggles ===
RUN_SMOKE = True
RUN_BASELINE_EVAL = True
RUN_DATA_BUILD = True
RUN_TRAIN_LEAD = True
RUN_TRAIN_AUDITOR = True
RUN_TRAINED_EVAL = True
RUN_PLOTS = True
PUSH_FINAL_ADAPTERS = True

# === Persistence ===
PERSIST_LOGS_TO_HUB = True
PERSIST_PLOTS_TO_HUB = True
PERSIST_EVERY_PHASE = True

# === Paths ===
BASE_DIR = Path("/content")
WORKTREE_DIR = BASE_DIR / "sdk_sovereign_pkg"
LOGS_DIR = BASE_DIR / "logs"
CHECKPOINTS_DIR = BASE_DIR / "checkpoints"
ARTIFACTS_DIR = BASE_DIR / "artifacts"
PLOTS_DIR = BASE_DIR / "plots"
DATA_DIR = BASE_DIR / "data"

for directory in [LOGS_DIR, CHECKPOINTS_DIR, ARTIFACTS_DIR, PLOTS_DIR, DATA_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
print(f"RUN_ID: {RUN_ID}")

# === Authenticate ===
hf_token = userdata.get("HF_TOKEN")
login(token=hf_token)
os.environ["HF_TOKEN"] = hf_token

wandb_key = userdata.get("WANDB_API_KEY")
if wandb_key:
    os.environ["WANDB_API_KEY"] = wandb_key
    import wandb

    wandb.login(key=wandb_key)
    print("OK: W&B authenticated")

# === Clone repo ===
if not WORKTREE_DIR.exists():
    subprocess.run(["git", "clone", f"https://github.com/{GH_REPO}.git", str(WORKTREE_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(WORKTREE_DIR), "pull"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(WORKTREE_DIR)], check=True)
if str(WORKTREE_DIR) not in sys.path:
    sys.path.insert(0, str(WORKTREE_DIR))

# === Create HF Hub repos ===
api = HfApi()
for repo_id in [CHECKPOINT_REPO, LEAD_ADAPTER_REPO, AUDITOR_ADAPTER_REPO]:
    try:
        create_repo(repo_id, repo_type="model", private=False, exist_ok=True)
        print(f"OK: repo ready -> {repo_id}")
    except Exception as exc:
        print(f"WARN: could not create {repo_id}: {exc}")

# === Persistent state ===
RUN_STATE_PATH = LOGS_DIR / "run_state_latest.json"
RUN_MANIFEST_PATH = LOGS_DIR / f"run_manifest_{RUN_ID}.json"
run_state = {
    "run_id": RUN_ID,
    "started_at": datetime.now().isoformat(),
    "gh_repo": GH_REPO,
    "hf_user": HF_USER,
    "checkpoint_repo": CHECKPOINT_REPO,
    "lead_adapter_repo": LEAD_ADAPTER_REPO,
    "auditor_adapter_repo": AUDITOR_ADAPTER_REPO,
    "config": {
        "expert_episodes_per_repo": EXPERT_EPISODES_PER_REPO,
        "negative_auditor_examples_per_repo": NEGATIVE_AUDITOR_EXAMPLES_PER_REPO,
        "n_baseline_eval_per_repo": N_BASELINE_EVAL_PER_REPO,
        "n_trained_eval_per_repo": N_TRAINED_EVAL_PER_REPO,
        "lead_sft_max_steps": LEAD_SFT_MAX_STEPS,
        "auditor_sft_max_steps": AUDITOR_SFT_MAX_STEPS,
        "lead_grpo_max_steps": LEAD_GRPO_MAX_STEPS,
        "auditor_grpo_max_steps": AUDITOR_GRPO_MAX_STEPS,
        "save_every_n_steps": SAVE_EVERY_N_STEPS,
    },
    "completed_phases": [],
}
RUN_STATE_PATH.write_text(json.dumps(run_state, indent=2))
RUN_MANIFEST_PATH.write_text(json.dumps(run_state, indent=2))

print(f"OK: config/auth complete. State file: {RUN_STATE_PATH}")

## Cell 3 — Start localhost env server (avoids WebSocket timeouts)

In [ ]:
# Cell 3 — Localhost env server + persistent loggers
import json
import shutil
import subprocess
import time
from datetime import datetime

import requests


def _json_ready(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(key): _json_ready(val) for key, val in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [_json_ready(item) for item in value]
    if hasattr(value, "item"):
        try:
            return value.item()
        except Exception:
            return str(value)
    return value


class JsonlLogger:
    """Append-only JSONL logger that flushes every row."""

    def __init__(self, path: Path):
        self.path = path
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self._fp = open(self.path, "a", buffering=1, encoding="utf-8")

    def log(self, **payload):
        payload = _json_ready(payload)
        payload["_ts"] = datetime.now().isoformat()
        self._fp.write(json.dumps(payload, ensure_ascii=False) + "\n")
        self._fp.flush()

    def close(self):
        try:
            self._fp.flush()
            self._fp.close()
        except Exception:
            pass


step_logger = JsonlLogger(LOGS_DIR / f"step_log_{RUN_ID}.jsonl")
metric_logger = JsonlLogger(LOGS_DIR / f"metric_log_{RUN_ID}.jsonl")
phase_logger = JsonlLogger(LOGS_DIR / f"phase_log_{RUN_ID}.jsonl")


def safe_step_log(event: str, **payload):
    try:
        step_logger.log(event=event, **payload)
    except Exception as exc:
        print(f"WARN: step logger unavailable: {exc}")


def safe_metric_log(event: str, **payload):
    try:
        metric_logger.log(event=event, **payload)
    except Exception as exc:
        print(f"WARN: metric logger unavailable: {exc}")


def update_run_state(phase_name: str, **extra):
    state = json.loads(RUN_STATE_PATH.read_text()) if RUN_STATE_PATH.exists() else dict(run_state)
    if phase_name not in state.get("completed_phases", []):
        state.setdefault("completed_phases", []).append(phase_name)
    state[phase_name] = _json_ready(extra)
    state["last_updated_at"] = datetime.now().isoformat()
    RUN_STATE_PATH.write_text(json.dumps(state, indent=2, ensure_ascii=False))
    RUN_MANIFEST_PATH.write_text(json.dumps(state, indent=2, ensure_ascii=False))
    phase_logger.log(event="phase_complete", phase=phase_name, details=extra)


def push_folder_safe(folder_path: Path, repo_id: str, commit_message: str, path_in_repo: str | None = None):
    if not folder_path.exists():
        return False
    try:
        upload_kwargs = {
            "folder_path": str(folder_path),
            "repo_id": repo_id,
            "repo_type": "model",
            "commit_message": commit_message,
        }
        if path_in_repo:
            upload_kwargs["path_in_repo"] = path_in_repo
        api.upload_folder(**upload_kwargs)
        safe_step_log("hub_upload_ok", repo_id=repo_id, path_in_repo=path_in_repo, folder=str(folder_path))
        return True
    except Exception as exc:
        print(f"WARN: upload failed for {repo_id}: {exc}")
        safe_step_log("hub_upload_fail", repo_id=repo_id, path_in_repo=path_in_repo, error=str(exc))
        return False


def push_file_safe(file_path: Path, repo_id: str, commit_message: str, path_in_repo: str):
    if not file_path.exists():
        return False
    try:
        api.upload_file(
            path_or_fileobj=str(file_path),
            path_in_repo=path_in_repo,
            repo_id=repo_id,
            repo_type="model",
            commit_message=commit_message,
        )
        safe_step_log("hub_file_ok", repo_id=repo_id, path_in_repo=path_in_repo)
        return True
    except Exception as exc:
        print(f"WARN: file upload failed for {file_path}: {exc}")
        safe_step_log("hub_file_fail", repo_id=repo_id, path=str(file_path), error=str(exc))
        return False


def persist_phase_snapshot(phase_name: str):
    if not PERSIST_EVERY_PHASE:
        return
    if PERSIST_LOGS_TO_HUB:
        push_folder_safe(LOGS_DIR, CHECKPOINT_REPO, f"{phase_name} logs {RUN_ID}", path_in_repo=f"runs/{RUN_ID}/logs")
        push_file_safe(RUN_STATE_PATH, CHECKPOINT_REPO, f"{phase_name} state {RUN_ID}", path_in_repo=f"runs/{RUN_ID}/run_state_latest.json")


subprocess.run(["pkill", "-f", "uvicorn"], check=False)
time.sleep(2)

candidate_modules = [
    "server.app:app",
    "sdk_sovereign.server.app:app",
    "openenv_sdk_sovereign.server.app:app",
]

env_proc = None
ENV_URL = "https://ishansurdi-sdk-sovereign.hf.space"
for module_path in candidate_modules:
    try:
        proc = subprocess.Popen(
            ["uvicorn", module_path, "--host", "0.0.0.0", "--port", "8000"],
            cwd=str(WORKTREE_DIR),
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
        )
        time.sleep(8)
        response = requests.get("http://localhost:8000/health", timeout=5)
        if response.status_code == 200:
            env_proc = proc
            ENV_URL = "http://localhost:8000"
            print(f"OK: local env server running with {module_path}")
            break
        proc.terminate()
        time.sleep(1)
    except Exception:
        try:
            proc.terminate()
            time.sleep(1)
        except Exception:
            pass

safe_step_log("env_ready", env_url=ENV_URL)
print(f"ENV_URL = {ENV_URL}")
print(f"Logs will persist under: {LOGS_DIR}")

## Cell 4 — Load model, agents, and HF Hub helpers

In [ ]:
# Cell 4 — Load model + helpers
import importlib
import json
import re
import shutil
from collections import Counter, defaultdict
from statistics import mean

import torch
from datasets import Dataset
from huggingface_hub import hf_hub_download
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments, TrainerCallback

import server.llm_agents as la
from client import SDKSovereignEnv
from models import SDKAction, SDKObservation
from scripts.hand_patches import GOOD_PATCHES
from server.environment import SDKSovereignEnvironment
from server.rule_agents import auditor_rule_agent, lead_rule_agent
from server.verifier import Verifier

importlib.reload(la)

REPO_ORDER = ["payments_repo", "maps_repo", "comms_repo"]
REPO_TO_GT = {
    "payments_repo": "razorpay",
    "maps_repo": "mmi_sdk",
    "comms_repo": "kaleyra",
}
WRONG_PROPOSALS = {
    "payments_repo": "foreign_payments_sdk",
    "maps_repo": "foreign_maps_sdk",
    "comms_repo": "foreign_sms_sdk",
}


def load_grpo_symbols():
    from trl.trainer.grpo_trainer import GRPOConfig, GRPOTrainer

    return GRPOTrainer, GRPOConfig


def init_wandb_run(run_name: str, config: dict | None = None):
    import wandb

    kwargs = {
        "project": WANDB_PROJECT,
        "name": f"{run_name}_{RUN_ID}",
        "config": config or {},
        "reinit": True,
    }
    if WANDB_ENTITY:
        kwargs["entity"] = WANDB_ENTITY
    return wandb.init(**kwargs)


verifier = Verifier(WORKTREE_DIR / "server" / "repos")


def make_model_and_agents():
    model, tokenizer = la.load_model_with_two_adapters()
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if hasattr(model.config, "use_cache"):
        model.config.use_cache = False
    return model, tokenizer, la.make_agent_pair(model, tokenizer)


model, tokenizer, agents = make_model_and_agents()
print(f"Loaded model. CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


def action_to_json(action: SDKAction) -> str:
    payload = {}
    for key, value in action.__dict__.items():
        if key == "role" or value is None:
            continue
        payload[key] = value
    return json.dumps(payload, ensure_ascii=False)


def prompt_plus_completion(prompt: str, completion: str) -> str:
    return f"{prompt}{completion}"


def save_jsonl(path: Path, rows: list[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text("\n".join(json.dumps(_json_ready(row), ensure_ascii=False) for row in rows), encoding="utf-8")


def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def build_local_env(seed: int = 0) -> SDKSovereignEnvironment:
    return SDKSovereignEnvironment(repos_root=WORKTREE_DIR / "server" / "repos", seed=seed)


def get_state(env) -> dict:
    state_attr = getattr(env, "state")
    return state_attr() if callable(state_attr) else state_attr


def infer_repo_from_prompt(prompt: str) -> str:
    lower_prompt = prompt.lower()
    if "charge_customer" in lower_prompt or "stripe" in lower_prompt:
        return "payments_repo"
    if "address_to_coords" in lower_prompt or "googlemaps" in lower_prompt:
        return "maps_repo"
    return "comms_repo"


def extract_prompt_line(prompt: str, label: str) -> str | None:
    match = re.search(rf"{re.escape(label)}\s*(.+)", prompt)
    return match.group(1).strip() if match else None


def completion_to_text(completion) -> str:
    if isinstance(completion, str):
        return completion
    if isinstance(completion, list):
        parts = []
        for item in completion:
            if isinstance(item, str):
                parts.append(item)
            elif isinstance(item, dict):
                parts.append(str(item.get("content", "")))
            else:
                parts.append(str(item))
        return "".join(parts)
    if isinstance(completion, dict):
        return str(completion.get("content", ""))
    return str(completion)


def set_only_adapter_trainable(model, adapter_name: str):
    model.set_adapter(adapter_name)
    trainable = 0
    total = 0
    for name, param in model.named_parameters():
        total += param.numel()
        param.requires_grad = adapter_name in name
        if param.requires_grad:
            trainable += param.numel()
    print(f"Trainable params for {adapter_name}: {trainable:,} / {total:,}")


def build_causal_lm_dataset(text_rows: list[str], max_length: int):
    encoded_rows = []
    for text in text_rows:
        encoded = tokenizer(
            text,
            truncation=True,
            max_length=max_length,
            padding="max_length",
        )
        encoded["labels"] = list(encoded["input_ids"])
        encoded_rows.append(encoded)
    return Dataset.from_list(encoded_rows)


class AdapterPersistenceCallback(TrainerCallback):
    def __init__(self, adapter_name: str, repo_id: str, phase_name: str):
        self.adapter_name = adapter_name
        self.repo_id = repo_id
        self.phase_name = phase_name
        self.last_saved_step = -1

    def _save_snapshot(self, model_obj, step_value):
        if model_obj is None:
            return
        snapshot_dir = ARTIFACTS_DIR / f"{self.phase_name}_step_{step_value}"
        snapshot_dir.mkdir(parents=True, exist_ok=True)
        model_obj.set_adapter(self.adapter_name)
        try:
            model_obj.save_pretrained(str(snapshot_dir), selected_adapters=[self.adapter_name])
        except TypeError:
            model_obj.save_pretrained(str(snapshot_dir))
        push_folder_safe(snapshot_dir, self.repo_id, f"{self.phase_name} step {step_value}", path_in_repo=f"step_{step_value}")
        shutil.rmtree(snapshot_dir, ignore_errors=True)

    def on_log(self, args, state, control, logs=None, **kwargs):
        safe_metric_log(event=f"{self.phase_name}_trainer_log", step=int(state.global_step), logs=logs or {})

    def on_save(self, args, state, control, **kwargs):
        if state.global_step != self.last_saved_step:
            self.last_saved_step = state.global_step
            self._save_snapshot(kwargs.get("model"), state.global_step)

    def on_train_end(self, args, state, control, **kwargs):
        self._save_snapshot(kwargs.get("model"), f"final_{state.global_step}")


def save_single_adapter(model_obj, adapter_name: str, target_dir: Path):
    target_dir.mkdir(parents=True, exist_ok=True)
    model_obj.set_adapter(adapter_name)
    try:
        model_obj.save_pretrained(str(target_dir), selected_adapters=[adapter_name])
    except TypeError:
        model_obj.save_pretrained(str(target_dir))


def load_eval_adapter(model_obj, repo_id: str, adapter_name: str) -> tuple[bool, str]:
    tried = []
    try:
        model_obj.load_adapter(repo_id, adapter_name=adapter_name)
        return True, f"hub-root:{repo_id}"
    except Exception as exc:
        tried.append(str(exc))
    try:
        repo_files = api.list_repo_files(repo_id, repo_type="model")
        step_dirs = sorted({part.split("/")[0] for part in repo_files if part.startswith("step_")})
        for step_dir in reversed(step_dirs):
            try:
                local_dir = ARTIFACTS_DIR / f"{adapter_name}_{step_dir}"
                local_dir.mkdir(parents=True, exist_ok=True)
                for filename in ["adapter_config.json", "adapter_model.safetensors", "README.md"]:
                    try:
                        downloaded = hf_hub_download(repo_id=repo_id, repo_type="model", filename=f"{step_dir}/{filename}")
                        shutil.copy(downloaded, local_dir / filename)
                    except Exception:
                        pass
                model_obj.load_adapter(str(local_dir), adapter_name=adapter_name)
                return True, f"hub-backup:{repo_id}/{step_dir}"
            except Exception as exc:
                tried.append(str(exc))
    except Exception as exc:
        tried.append(str(exc))
    return False, " | ".join(tried[-3:])


def summarize_results(results: list[dict]) -> dict:
    if not results:
        return {"episodes": 0, "pass_rate": 0.0, "mean_reward": 0.0}
    repo_counts = defaultdict(list)
    action_counts = Counter()
    for row in results:
        repo_counts[row["repo"]].append(bool(row["success"]))
        for turn in row.get("transcript", []):
            action_counts[f"{turn['role']}::{turn['action_type']}"] += 1
    return {
        "episodes": len(results),
        "pass_rate": sum(bool(row["success"]) for row in results) / len(results),
        "mean_reward": mean(float(row["total_reward"]) for row in results),
        "mean_turns": mean(int(row["turns"]) for row in results),
        "repo_pass_rate": {
            repo_id: sum(values) / len(values) for repo_id, values in repo_counts.items()
        },
        "action_counts": dict(action_counts),
    }


def evaluate_agents(agent_pair: dict, n_per_repo: int, phase_name: str) -> list[dict]:
    rows = []
    for repo_id in REPO_ORDER:
        for offset in range(n_per_repo):
            seed = 1000 + (len(rows) * 17) + offset
            env = build_local_env(seed=seed)
            obs = env.reset(seed=seed, repo_id=repo_id, episode_id=f"{phase_name}_{repo_id}_{offset}")
            transcript = []
            total_reward = 0.0
            while not obs.done and obs.turn_index < 7:
                action = agent_pair[obs.current_role](obs)
                transcript.append(
                    {
                        "turn": int(obs.turn_index),
                        "role": obs.current_role,
                        "action_type": action.action_type,
                        "proposed_sdk": action.proposed_sdk,
                        "approved_replacement": obs.approved_replacement,
                    }
                )
                obs = env.step(action)
                total_reward += float(obs.reward)
            state = get_state(env)
            test_results = state.test_results or {}
            row = {
                "phase": phase_name,
                "repo": repo_id,
                "episode_seed": seed,
                "total_reward": float(total_reward),
                "success": bool(test_results and all(test_results.values())),
                "tests_passed": int(sum(bool(value) for value in test_results.values())) if test_results else 0,
                "tests_total": int(len(test_results)) if test_results else 3,
                "turns": int(state.step_count),
                "terminated": state.terminated_reason,
                "reward_by_role": dict(state.cumulative_reward_by_role),
                "transcript": transcript,
            }
            rows.append(row)
            safe_step_log(f"{phase_name}_episode", **row)
    return rows


safe_step_log("model_loaded", adapters=list(model.peft_config.keys()))
print("Helpers ready.")

## Cell 5 — Smoke test (verify env + model + adapters work)

In [ ]:
# Cell 5 — Smoke test
if RUN_SMOKE:
    smoke_rows = []
    for repo_id in REPO_ORDER:
        env = build_local_env(seed=7)
        obs = env.reset(seed=7, repo_id=repo_id, episode_id=f"smoke_{repo_id}")
        transcript = []
        while not obs.done and obs.turn_index < 7:
            action = agents[obs.current_role](obs)
            transcript.append(
                {
                    "turn": int(obs.turn_index),
                    "role": obs.current_role,
                    "action_type": action.action_type,
                    "proposed_sdk": action.proposed_sdk,
                }
            )
            obs = env.step(action)
        state = get_state(env)
        row = {
            "repo": repo_id,
            "success": bool(state.test_results and all(state.test_results.values())),
            "terminated": state.terminated_reason,
            "reward_by_role": dict(state.cumulative_reward_by_role),
            "transcript": transcript,
        }
        smoke_rows.append(row)
        print(f"Smoke {repo_id}: success={row['success']} terminated={row['terminated']}")
        safe_step_log("smoke_result", **row)

    dummy_auditor_obs = SDKObservation(
        current_role="auditor",
        turn_index=0,
        max_turns=7,
        error_log="test",
        conversation_history=[],
        visible_codebase=None,
        visible_filename=None,
        visible_allowlist=["razorpay"],
        current_proposal=None,
        approved_replacement=None,
        done=False,
        reward=0.0,
        reward_breakdown={},
    )
    agents["auditor"](dummy_auditor_obs)
    assert model.active_adapter == "auditor_adapter"

    dummy_lead_obs = SDKObservation(
        current_role="lead",
        turn_index=1,
        max_turns=7,
        error_log="test",
        conversation_history=[],
        visible_codebase="import stripe",
        visible_filename="broken.py",
        visible_allowlist=None,
        current_proposal=None,
        approved_replacement=None,
        done=False,
        reward=0.0,
        reward_breakdown={},
    )
    agents["lead"](dummy_lead_obs)
    assert model.active_adapter == "lead_adapter"

    save_jsonl(LOGS_DIR / f"smoke_results_{RUN_ID}.jsonl", smoke_rows)
    update_run_state("smoke", smoke_rows=smoke_rows)
    persist_phase_snapshot("smoke")
    print("OK: smoke completed")
else:
    smoke_rows = []
    print("Skipped smoke test")

## Cell 6 — Baseline evaluation (untrained model)

In [ ]:
# Cell 6 — Baseline evaluation
if RUN_BASELINE_EVAL:
    baseline_results = evaluate_agents(agents, n_per_repo=N_BASELINE_EVAL_PER_REPO, phase_name="baseline")
    baseline_summary = summarize_results(baseline_results)
    baseline_jsonl = LOGS_DIR / f"baseline_results_{RUN_ID}.jsonl"
    baseline_summary_path = LOGS_DIR / f"baseline_summary_{RUN_ID}.json"
    save_jsonl(baseline_jsonl, baseline_results)
    baseline_summary_path.write_text(json.dumps(baseline_summary, indent=2, ensure_ascii=False), encoding="utf-8")
    update_run_state("baseline_eval", summary=baseline_summary)
    persist_phase_snapshot("baseline_eval")
    print(json.dumps(baseline_summary, indent=2))
else:
    baseline_results = []
    baseline_summary = {"episodes": 0, "pass_rate": 0.0, "mean_reward": 0.0}
    print("Skipped baseline evaluation")

## Cell 7 — Build expert datasets and prompt pools

In [ ]:
# Cell 7 — Build fast expert datasets and GRPO prompt pools
if RUN_DATA_BUILD:
    expert_lead_rows = []
    expert_auditor_rows = []

    for repo_id in REPO_ORDER:
        for repeat_idx in range(EXPERT_EPISODES_PER_REPO):
            env = build_local_env(seed=repeat_idx)
            obs = env.reset(seed=repeat_idx, repo_id=repo_id, episode_id=f"expert_{repo_id}_{repeat_idx}")

            auditor_wait = auditor_rule_agent(obs)
            expert_auditor_rows.append(
                {
                    "repo": repo_id,
                    "kind": "wait",
                    "prompt": agents["auditor"]._build_prompt(obs),
                    "completion": action_to_json(auditor_wait),
                    "text": prompt_plus_completion(agents["auditor"]._build_prompt(obs), action_to_json(auditor_wait)),
                }
            )
            obs = env.step(auditor_wait)

            lead_propose = lead_rule_agent(obs)
            expert_lead_rows.append(
                {
                    "repo": repo_id,
                    "kind": "propose",
                    "prompt": agents["lead"]._build_prompt(obs),
                    "completion": action_to_json(lead_propose),
                    "text": prompt_plus_completion(agents["lead"]._build_prompt(obs), action_to_json(lead_propose)),
                }
            )
            obs = env.step(lead_propose)

            auditor_approve = auditor_rule_agent(obs)
            expert_auditor_rows.append(
                {
                    "repo": repo_id,
                    "kind": "approve",
                    "prompt": agents["auditor"]._build_prompt(obs),
                    "completion": action_to_json(auditor_approve),
                    "text": prompt_plus_completion(agents["auditor"]._build_prompt(obs), action_to_json(auditor_approve)),
                }
            )
            obs = env.step(auditor_approve)

            lead_submit = lead_rule_agent(obs)
            expert_lead_rows.append(
                {
                    "repo": repo_id,
                    "kind": "submit",
                    "prompt": agents["lead"]._build_prompt(obs),
                    "completion": action_to_json(lead_submit),
                    "text": prompt_plus_completion(agents["lead"]._build_prompt(obs), action_to_json(lead_submit)),
                }
            )

        for negative_idx in range(NEGATIVE_AUDITOR_EXAMPLES_PER_REPO):
            env = build_local_env(seed=800 + negative_idx)
            obs = env.reset(seed=800 + negative_idx, repo_id=repo_id, episode_id=f"negative_{repo_id}_{negative_idx}")
            obs = env.step(SDKAction(role="auditor", action_type="pass", reasoning="Waiting for proposal."))
            obs = env.step(
                SDKAction(
                    role="lead",
                    action_type="propose_replacement",
                    proposed_sdk=WRONG_PROPOSALS[repo_id],
                    reasoning="Trying a non-sovereign SDK.",
                )
            )
            reject_action = SDKAction(
                role="auditor",
                action_type="reject",
                rejection_reason=f"{WRONG_PROPOSALS[repo_id]} is not on the sovereign allow-list.",
                reasoning="Rejecting because the proposal is not on the allow-list.",
            )
            expert_auditor_rows.append(
                {
                    "repo": repo_id,
                    "kind": "reject",
                    "prompt": agents["auditor"]._build_prompt(obs),
                    "completion": action_to_json(reject_action),
                    "text": prompt_plus_completion(agents["auditor"]._build_prompt(obs), action_to_json(reject_action)),
                }
            )

    lead_dataset_path = DATA_DIR / f"expert_lead_{RUN_ID}.jsonl"
    auditor_dataset_path = DATA_DIR / f"expert_auditor_{RUN_ID}.jsonl"
    save_jsonl(lead_dataset_path, expert_lead_rows)
    save_jsonl(auditor_dataset_path, expert_auditor_rows)

    lead_sft_dataset = build_causal_lm_dataset([row["text"] for row in expert_lead_rows], max_length=1024)
    auditor_sft_dataset = build_causal_lm_dataset([row["text"] for row in expert_auditor_rows], max_length=768)
    lead_grpo_prompts = Dataset.from_dict({"prompt": [row["prompt"] for row in expert_lead_rows]})
    auditor_grpo_prompts = Dataset.from_dict({"prompt": [row["prompt"] for row in expert_auditor_rows]})

    data_summary = {
        "lead_examples": len(expert_lead_rows),
        "auditor_examples": len(expert_auditor_rows),
        "lead_prompt_kinds": dict(Counter(row["kind"] for row in expert_lead_rows)),
        "auditor_prompt_kinds": dict(Counter(row["kind"] for row in expert_auditor_rows)),
    }
    update_run_state("data_build", summary=data_summary)
    persist_phase_snapshot("data_build")
    print(json.dumps(data_summary, indent=2))
else:
    expert_lead_rows = []
    expert_auditor_rows = []
    lead_sft_dataset = None
    auditor_sft_dataset = None
    lead_grpo_prompts = None
    auditor_grpo_prompts = None
    print("Skipped data build")

## Cell 8 — Train Lead adapter fast

In [ ]:
# Cell 8 — Train Lead adapter fast: bootstrap SFT + tiny GRPO refine
if RUN_TRAIN_LEAD and lead_sft_dataset is not None:
    import wandb

    init_wandb_run(
        "lead_fast_pipeline",
        config={
            "lead_sft_max_steps": LEAD_SFT_MAX_STEPS,
            "lead_grpo_max_steps": LEAD_GRPO_MAX_STEPS,
            "lead_examples": len(expert_lead_rows),
        },
    )

    set_only_adapter_trainable(model, "lead_adapter")
    lead_callback = AdapterPersistenceCallback("lead_adapter", LEAD_ADAPTER_REPO, "lead")
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    lead_sft_args = TrainingArguments(
        output_dir=str(CHECKPOINTS_DIR / "lead_sft"),
        max_steps=LEAD_SFT_MAX_STEPS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        save_strategy="steps",
        report_to="wandb",
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )
    lead_sft_trainer = Trainer(
        model=model,
        args=lead_sft_args,
        train_dataset=lead_sft_dataset,
        data_collator=collator,
        callbacks=[lead_callback],
    )
    print("Starting Lead SFT bootstrap...")
    lead_sft_trainer.train()

    GRPOTrainer, GRPOConfig = load_grpo_symbols()

    def lead_reward_fn(completions, prompts=None, **kwargs):
        prompt_source = kwargs.get("prompts") or kwargs.get("prompt") or prompts or [""] * len(completions)
        if isinstance(prompt_source, str):
            prompt_source = [prompt_source] * len(completions)
        rewards = []
        for idx, completion in enumerate(completions):
            prompt_text = str(prompt_source[idx]) if idx < len(prompt_source) else ""
            repo_id = infer_repo_from_prompt(prompt_text)
            gt_sdk = REPO_TO_GT[repo_id]
            approved = extract_prompt_line(prompt_text, "APPROVED REPLACEMENT:") or ""
            approved = None if approved.startswith("(not yet approved)") else approved
            action = agents["lead"]._parse_action(completion_to_text(completion))
            reward = 0.0
            if action.action_type in {"propose_replacement", "submit_patch", "request_hint", "pass"}:
                reward += 0.25
            if action.action_type == "propose_replacement":
                reward += 1.0 if action.proposed_sdk == gt_sdk else -0.75
                if (action.reasoning or ""):
                    reward += 0.05
            elif action.action_type == "submit_patch":
                if not approved:
                    reward -= 1.5
                code = action.patched_code or ""
                if verifier.syntax_ok(code):
                    reward += 1.0
                    imports = verifier.extract_imports(code)
                    if approved and approved in imports:
                        reward += 1.5
                    elif gt_sdk in imports:
                        reward += 1.0
                    else:
                        reward -= 2.0
                    test_results = verifier.run_parity_tests(code, repo_id)
                    passed = sum(bool(value) for value in test_results.values())
                    reward += 1.5 * passed
                    if test_results and all(test_results.values()):
                        reward += 4.0
                else:
                    reward -= 1.0
            elif action.action_type == "request_hint":
                reward += 0.15
            elif action.action_type == "pass":
                reward -= 0.6
            else:
                reward -= 0.4
            rewards.append(float(reward))
        safe_metric_log("lead_reward_batch", mean_reward=sum(rewards) / max(len(rewards), 1), rewards_preview=rewards[:8], batch_size=len(rewards))
        return rewards

    set_only_adapter_trainable(model, "lead_adapter")
    lead_grpo_args = GRPOConfig(
        output_dir=str(CHECKPOINTS_DIR / "lead_grpo"),
        max_steps=LEAD_GRPO_MAX_STEPS,
        num_generations=2,
        max_completion_length=384,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        learning_rate=5e-6,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        report_to="wandb",
    )
    lead_grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=lead_reward_fn,
        args=lead_grpo_args,
        train_dataset=lead_grpo_prompts.select(range(min(len(lead_grpo_prompts), 24))),
        tokenizer=tokenizer,
        callbacks=[lead_callback],
    )
    print("Starting tiny Lead GRPO refine...")
    lead_grpo_trainer.train()

    lead_final_dir = ARTIFACTS_DIR / "lead_adapter_final"
    save_single_adapter(model, "lead_adapter", lead_final_dir)
    if PUSH_FINAL_ADAPTERS:
        push_folder_safe(lead_final_dir, LEAD_ADAPTER_REPO, f"lead final {RUN_ID}")

    wandb.finish()
    update_run_state(
        "lead_training",
        sft_steps=LEAD_SFT_MAX_STEPS,
        grpo_steps=LEAD_GRPO_MAX_STEPS,
        final_dir=str(lead_final_dir),
    )
    persist_phase_snapshot("lead_training")
    print("OK: Lead training complete")
else:
    print("Skipped Lead training")

## Cell 9 — Train Auditor adapter fast

In [ ]:
# Cell 9 — Train Auditor adapter fast: bootstrap SFT + tiny GRPO refine
if RUN_TRAIN_AUDITOR and auditor_sft_dataset is not None:
    import wandb

    init_wandb_run(
        "auditor_fast_pipeline",
        config={
            "auditor_sft_max_steps": AUDITOR_SFT_MAX_STEPS,
            "auditor_grpo_max_steps": AUDITOR_GRPO_MAX_STEPS,
            "auditor_examples": len(expert_auditor_rows),
        },
    )

    set_only_adapter_trainable(model, "auditor_adapter")
    auditor_callback = AdapterPersistenceCallback("auditor_adapter", AUDITOR_ADAPTER_REPO, "auditor")
    collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

    auditor_sft_args = TrainingArguments(
        output_dir=str(CHECKPOINTS_DIR / "auditor_sft"),
        max_steps=AUDITOR_SFT_MAX_STEPS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        save_strategy="steps",
        report_to="wandb",
        remove_unused_columns=False,
        fp16=torch.cuda.is_available(),
        dataloader_num_workers=0,
    )
    auditor_sft_trainer = Trainer(
        model=model,
        args=auditor_sft_args,
        train_dataset=auditor_sft_dataset,
        data_collator=collator,
        callbacks=[auditor_callback],
    )
    print("Starting Auditor SFT bootstrap...")
    auditor_sft_trainer.train()

    GRPOTrainer, GRPOConfig = load_grpo_symbols()

    def auditor_reward_fn(completions, prompts=None, **kwargs):
        prompt_source = kwargs.get("prompts") or kwargs.get("prompt") or prompts or [""] * len(completions)
        if isinstance(prompt_source, str):
            prompt_source = [prompt_source] * len(completions)
        rewards = []
        for idx, completion in enumerate(completions):
            prompt_text = str(prompt_source[idx]) if idx < len(prompt_source) else ""
            repo_id = infer_repo_from_prompt(prompt_text)
            gt_sdk = REPO_TO_GT[repo_id]
            proposal = extract_prompt_line(prompt_text, "CURRENT PROPOSAL:") or ""
            proposal = None if proposal.startswith("(none yet)") else proposal
            allowlist_line = extract_prompt_line(prompt_text, "ALLOW-LIST:") or ""
            allowlist = {item.strip() for item in allowlist_line.split(",") if item.strip()}
            action = agents["auditor"]._parse_action(completion_to_text(completion))
            reward = 0.0
            if action.action_type in {"approve", "reject", "give_hint", "pass"}:
                reward += 0.25
            if not proposal:
                if action.action_type == "pass":
                    reward += 0.30
                elif action.action_type == "give_hint":
                    reward += 0.10 if (action.hint_response or "") else -0.10
                else:
                    reward -= 0.50
            else:
                if action.action_type == "approve":
                    if proposal == gt_sdk:
                        reward += 1.50
                    elif proposal in allowlist:
                        reward += 0.50
                    else:
                        reward -= 2.00
                elif action.action_type == "reject":
                    if proposal not in allowlist:
                        reward += 1.00
                    elif proposal == gt_sdk:
                        reward -= 1.00
                    else:
                        reward -= 0.40
                elif action.action_type == "give_hint":
                    hint_text = (action.hint_response or "").lower()
                    reward += 0.25 if (gt_sdk in hint_text or repo_id.split("_")[0] in hint_text) else -0.20
                elif action.action_type == "pass":
                    reward -= 0.50
                else:
                    reward -= 0.30
            rewards.append(float(reward))
        safe_metric_log("auditor_reward_batch", mean_reward=sum(rewards) / max(len(rewards), 1), rewards_preview=rewards[:8], batch_size=len(rewards))
        return rewards

    set_only_adapter_trainable(model, "auditor_adapter")
    auditor_grpo_args = GRPOConfig(
        output_dir=str(CHECKPOINTS_DIR / "auditor_grpo"),
        max_steps=AUDITOR_GRPO_MAX_STEPS,
        num_generations=2,
        max_completion_length=160,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=2,
        learning_rate=5e-6,
        logging_steps=LOG_EVERY_N_STEPS,
        save_steps=SAVE_EVERY_N_STEPS,
        report_to="wandb",
    )
    auditor_grpo_trainer = GRPOTrainer(
        model=model,
        reward_funcs=auditor_reward_fn,
        args=auditor_grpo_args,
        train_dataset=auditor_grpo_prompts.select(range(min(len(auditor_grpo_prompts), 24))),
        tokenizer=tokenizer,
        callbacks=[auditor_callback],
    )
    print("Starting tiny Auditor GRPO refine...")
    auditor_grpo_trainer.train()

    auditor_final_dir = ARTIFACTS_DIR / "auditor_adapter_final"
    save_single_adapter(model, "auditor_adapter", auditor_final_dir)
    if PUSH_FINAL_ADAPTERS:
        push_folder_safe(auditor_final_dir, AUDITOR_ADAPTER_REPO, f"auditor final {RUN_ID}")

    wandb.finish()
    update_run_state(
        "auditor_training",
        sft_steps=AUDITOR_SFT_MAX_STEPS,
        grpo_steps=AUDITOR_GRPO_MAX_STEPS,
        final_dir=str(auditor_final_dir),
    )
    persist_phase_snapshot("auditor_training")
    print("OK: Auditor training complete")
else:
    print("Skipped Auditor training")

## Cell 10 — Trained model evaluation

In [ ]:
# Cell 10 — Evaluate trained adapters with explicit per-repo coverage
if RUN_TRAINED_EVAL:
    trained_model, trained_tokenizer, trained_agents = make_model_and_agents()
    lead_ok, lead_source = load_eval_adapter(trained_model, LEAD_ADAPTER_REPO, "lead_adapter_trained")
    auditor_ok, auditor_source = load_eval_adapter(trained_model, AUDITOR_ADAPTER_REPO, "auditor_adapter_trained")

    if lead_ok and auditor_ok:
        trained_agents = la.make_agent_pair(trained_model, trained_tokenizer)
        trained_agents["lead"].adapter_name = "lead_adapter_trained"
        trained_agents["auditor"].adapter_name = "auditor_adapter_trained"
        adapter_source = {"lead": lead_source, "auditor": auditor_source}
    else:
        trained_agents = agents
        adapter_source = {
            "lead": lead_source if lead_ok else "in-memory current session",
            "auditor": auditor_source if auditor_ok else "in-memory current session",
        }

    trained_results = evaluate_agents(trained_agents, n_per_repo=N_TRAINED_EVAL_PER_REPO, phase_name="trained")
    trained_summary = summarize_results(trained_results)
    trained_summary["adapter_source"] = adapter_source

    eval_payload = {
        "baseline": baseline_results,
        "baseline_summary": baseline_summary,
        "trained": trained_results,
        "trained_summary": trained_summary,
    }
    eval_json_path = LOGS_DIR / f"eval_results_{RUN_ID}.json"
    eval_json_path.write_text(json.dumps(eval_payload, indent=2, ensure_ascii=False), encoding="utf-8")
    save_jsonl(LOGS_DIR / f"trained_results_{RUN_ID}.jsonl", trained_results)

    update_run_state("trained_eval", summary=trained_summary)
    persist_phase_snapshot("trained_eval")

    print("Trained summary:")
    print(json.dumps(trained_summary, indent=2))
    if baseline_results:
        print("\nDelta vs baseline:")
        print(f"  pass_rate: {trained_summary['pass_rate'] - baseline_summary['pass_rate']:+.2%}")
        print(f"  mean_reward: {trained_summary['mean_reward'] - baseline_summary['mean_reward']:+.3f}")
else:
    trained_results = []
    trained_summary = {"episodes": 0, "pass_rate": 0.0, "mean_reward": 0.0}
    print("Skipped trained evaluation")

## Cell 11 — Generate plots

In [ ]:
# Cell 11 — Generate judge-friendly plots and summaries
if RUN_PLOTS and baseline_results and trained_results:
    import matplotlib.pyplot as plt
    import numpy as np
    import pandas as pd
    import seaborn as sns

    sns.set_theme(style="whitegrid")
    PLOTS_DIR.mkdir(parents=True, exist_ok=True)

    baseline_df = pd.DataFrame(baseline_results)
    trained_df = pd.DataFrame(trained_results)
    baseline_df["label"] = "Baseline"
    trained_df["label"] = "Trained"
    combined_df = pd.concat([baseline_df, trained_df], ignore_index=True)

    def save_plot(filename: str):
        path = PLOTS_DIR / filename
        plt.tight_layout()
        plt.savefig(path, dpi=180, bbox_inches="tight")
        plt.close()
        return path

    # 1. Pass-rate headline
    plt.figure(figsize=(7, 5))
    pass_values = [baseline_summary["pass_rate"], trained_summary["pass_rate"]]
    bars = plt.bar(["Baseline", "Trained"], pass_values, color=["#7f8c8d", "#0f766e"])
    for bar, value in zip(bars, pass_values):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.015, f"{value:.1%}", ha="center", fontweight="bold")
    plt.ylim(0, 1.05)
    plt.ylabel("All-tests pass rate")
    plt.title("Did the agents actually solve migrations?")
    save_plot("01_pass_rate_headline.png")

    # 2. Mean reward comparison
    plt.figure(figsize=(7, 5))
    reward_values = [baseline_summary["mean_reward"], trained_summary["mean_reward"]]
    bars = plt.bar(["Baseline", "Trained"], reward_values, color=["#94a3b8", "#2563eb"])
    for bar, value in zip(bars, reward_values):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.05, f"{value:+.2f}", ha="center", fontweight="bold")
    plt.axhline(0, color="black", linewidth=0.8)
    plt.ylabel("Mean episode reward")
    plt.title("Reward moved with behavior")
    save_plot("02_mean_reward.png")

    # 3. Per-repo pass rate
    repo_plot_rows = []
    for label, summary in [("Baseline", baseline_summary), ("Trained", trained_summary)]:
        for repo_id, value in summary.get("repo_pass_rate", {}).items():
            repo_plot_rows.append({"label": label, "repo": repo_id, "pass_rate": value})
    repo_plot_df = pd.DataFrame(repo_plot_rows)
    plt.figure(figsize=(8, 5))
    sns.barplot(data=repo_plot_df, x="repo", y="pass_rate", hue="label", palette=["#94a3b8", "#0f766e"])
    plt.ylim(0, 1.05)
    plt.ylabel("Pass rate")
    plt.xlabel("")
    plt.title("Coverage across all repos")
    save_plot("03_per_repo_pass_rate.png")

    # 4. Cumulative success curve
    plt.figure(figsize=(8, 5))
    for label, frame, color in [("Baseline", baseline_df, "#64748b"), ("Trained", trained_df, "#2563eb")]:
        ordered = frame.reset_index(drop=True).copy()
        ordered["success_int"] = ordered["success"].astype(int)
        ordered["running_pass_rate"] = ordered["success_int"].expanding().mean()
        plt.plot(np.arange(1, len(ordered) + 1), ordered["running_pass_rate"], marker="o", label=label, color=color)
    plt.ylim(0, 1.05)
    plt.xlabel("Episode")
    plt.ylabel("Running pass rate")
    plt.title("Did training help consistently?")
    plt.legend()
    save_plot("04_cumulative_success_curve.png")

    # 5. Action mix
    action_rows = []
    for label, results in [("Baseline", baseline_results), ("Trained", trained_results)]:
        counts = Counter()
        for row in results:
            for turn in row["transcript"]:
                counts[f"{turn['role']}::{turn['action_type']}"] += 1
        for action_name, count in counts.items():
            action_rows.append({"label": label, "action": action_name, "count": count})
    action_df = pd.DataFrame(action_rows)
    plt.figure(figsize=(11, 5))
    sns.barplot(data=action_df, x="action", y="count", hue="label", palette=["#94a3b8", "#0f766e"])
    plt.xticks(rotation=35, ha="right")
    plt.xlabel("")
    plt.ylabel("Count")
    plt.title("Negotiation behavior changed")
    save_plot("05_action_mix.png")

    # 6. Successful completion turns
    success_turns = combined_df[combined_df["success"]].copy()
    plt.figure(figsize=(8, 5))
    if not success_turns.empty:
        sns.histplot(data=success_turns, x="turns", hue="label", multiple="dodge", shrink=0.9, bins=np.arange(1, 9) - 0.5, palette=["#94a3b8", "#0f766e"])
    plt.xlabel("Turns used")
    plt.title("How quickly successful episodes finish")
    save_plot("06_success_turns_hist.png")

    # 7. Lead training curve from local metrics
    metric_rows = load_jsonl(LOGS_DIR / f"metric_log_{RUN_ID}.jsonl")
    lead_metric_rows = []
    auditor_metric_rows = []
    for row in metric_rows:
        logs = row.get("logs", {}) if isinstance(row.get("logs", {}), dict) else {}
        if row.get("event", "").startswith("lead"):
            lead_metric_rows.append({"step": row.get("step", 0), **logs})
        if row.get("event", "").startswith("auditor"):
            auditor_metric_rows.append({"step": row.get("step", 0), **logs})

    if lead_metric_rows:
        lead_metric_df = pd.DataFrame(lead_metric_rows).sort_values("step")
        lead_value_col = "loss" if "loss" in lead_metric_df.columns else ("reward" if "reward" in lead_metric_df.columns else None)
        if lead_value_col:
            plt.figure(figsize=(8, 5))
            plt.plot(lead_metric_df["step"], lead_metric_df[lead_value_col], marker="o", color="#1d4ed8")
            plt.xlabel("Step")
            plt.ylabel(lead_value_col)
            plt.title("Lead training signal")
            save_plot("07_lead_training_curve.png")

    if auditor_metric_rows:
        auditor_metric_df = pd.DataFrame(auditor_metric_rows).sort_values("step")
        auditor_value_col = "loss" if "loss" in auditor_metric_df.columns else ("reward" if "reward" in auditor_metric_df.columns else None)
        if auditor_value_col:
            plt.figure(figsize=(8, 5))
            plt.plot(auditor_metric_df["step"], auditor_metric_df[auditor_value_col], marker="o", color="#0f766e")
            plt.xlabel("Step")
            plt.ylabel(auditor_value_col)
            plt.title("Auditor training signal")
            save_plot("08_auditor_training_curve.png")

    judge_summary = {
        "baseline_summary": baseline_summary,
        "trained_summary": trained_summary,
        "uplift": {
            "pass_rate_delta": trained_summary["pass_rate"] - baseline_summary["pass_rate"],
            "mean_reward_delta": trained_summary["mean_reward"] - baseline_summary["mean_reward"],
        },
        "adapter_source": trained_summary.get("adapter_source", {}),
    }
    (PLOTS_DIR / f"judge_summary_{RUN_ID}.json").write_text(json.dumps(judge_summary, indent=2, ensure_ascii=False), encoding="utf-8")

    if PERSIST_PLOTS_TO_HUB:
        push_folder_safe(PLOTS_DIR, CHECKPOINT_REPO, f"plots {RUN_ID}", path_in_repo=f"runs/{RUN_ID}/plots")
    update_run_state("plots", files=[path.name for path in sorted(PLOTS_DIR.glob("*.png"))])
    persist_phase_snapshot("plots")
    print(f"OK: plots saved under {PLOTS_DIR}")
else:
    print("Skipped plots")

## Cell 12 — Final bundle: download everything as one archive

In [ ]:
# Cell 12 — Final bundle and permanent backup
import shutil
from google.colab import files

bundle_dir = ARTIFACTS_DIR / f"sdk_sovereign_bundle_{RUN_ID}"
bundle_dir.mkdir(parents=True, exist_ok=True)

for name, source_dir in {
    "logs": LOGS_DIR,
    "data": DATA_DIR,
    "plots": PLOTS_DIR,
    "checkpoints": CHECKPOINTS_DIR,
    "artifacts": ARTIFACTS_DIR,
}.items():
    if source_dir.exists():
        shutil.copytree(source_dir, bundle_dir / name, dirs_exist_ok=True)

manifest_rows = [
    f"run_id: {RUN_ID}",
    f"created_at: {datetime.now().isoformat()}",
    f"checkpoint_repo: {CHECKPOINT_REPO}",
    f"lead_adapter_repo: {LEAD_ADAPTER_REPO}",
    f"auditor_adapter_repo: {AUDITOR_ADAPTER_REPO}",
    "",
]
for file_path in sorted(bundle_dir.rglob("*")):
    if file_path.is_file():
        manifest_rows.append(f"{file_path.relative_to(bundle_dir)} | {file_path.stat().st_size / 1e6:.2f} MB")
(bundle_dir / "MANIFEST.txt").write_text("\n".join(manifest_rows), encoding="utf-8")

archive_path = BASE_DIR / f"sdk_sovereign_bundle_{RUN_ID}.tar.gz"
subprocess.run(["tar", "-czf", str(archive_path), "-C", str(ARTIFACTS_DIR.parent), bundle_dir.name], check=True)

push_folder_safe(bundle_dir, CHECKPOINT_REPO, f"final bundle {RUN_ID}", path_in_repo=f"runs/{RUN_ID}/bundle")
push_file_safe(archive_path, CHECKPOINT_REPO, f"bundle archive {RUN_ID}", path_in_repo=f"runs/{RUN_ID}/sdk_sovereign_bundle_{RUN_ID}.tar.gz")
update_run_state("final_bundle", archive=str(archive_path), bundle_dir=str(bundle_dir))
persist_phase_snapshot("final_bundle")

print("Permanent locations:")
print(f"  HF logs/bundle repo: https://huggingface.co/{CHECKPOINT_REPO}")
print(f"  Lead adapter repo: https://huggingface.co/{LEAD_ADAPTER_REPO}")
print(f"  Auditor adapter repo: https://huggingface.co/{AUDITOR_ADAPTER_REPO}")
print(f"  Local archive: {archive_path}")

try:
    files.download(str(archive_path))
except Exception as exc:
    print(f"Browser download skipped: {exc}")

step_logger.close()
metric_logger.close()
phase_logger.close()
print("ALL DONE")